In [ ]:
from common import *

## 9. Priprema podataka

Kako ćemo u fazi imputacije trenirati modele za popunjavanje nedostajućih vrednosti, već u ovoj fazi potrebno je izvršiti feature engineering kako bismo iz dataseta izvukli maskimalnu količinu informacija.

### 9.1 Transformacija postojicih podataka

U ovom delu izvrsicemo tranformaciju podataka koji vec postoje u nasem dataframe-u na onaj nacin koji ce nasim modelima pruziti maksimalnu kolicinu informacija.

#### 9.1.1 Vremenski podaci

Za uspešnu primenu algoritama mašinskog učenja, neophodno je adekvatno pretprocesirati vremenske komponente. Ukoliko bismo datume tretirali kao klasične kategorijske promenljive i primenili 'One-Hot Encoding', to bi dovelo do eksponencijalnog rasta dimenzionalnosti skupa podataka. Sa druge strane, 'Label Encoding' unosi problem linearne progresije, gde model gubi svest o cikličnosti pri prelasku sa kraja na početak vremenskog ciklusa.

Zbog toga je logičnije primeniti metodu kružnog enkodiranja (Cyclical Encoding), transformacijom dana i meseci u tačke na dvodimenzionalnoj kružnici upotrebom sinusnih i kosinusnih funkcija. Godinu tretiramo kao nezavisnu promenljivu, ali s obzirom na to da su njene apsolutne vrednosti (2007-2017) neuporedivo veće od vrednosti sinusa i kosinusa (-1 do 1), primenićemo skaliranje godina na interval od 0 do 1. Ovo osigurava da disproporcija u magnitudama ne dovede do toga da godina dominira nad ostalim vremenskim faktorima prilikom učenja modela.

In [ ]:
data = loadData('backups/weatherAusAfter8_3.csv')
is_train = data['Date'] < SPLIT_DATE  # ponovo racunamo posle reload-a (indeks/broj redova se promenio)

data['Year_Scaled'] = (data['Year'] - data.loc[is_train, 'Year'].min()) / (data.loc[is_train, 'Year'].max() - data.loc[is_train, 'Year'].min())

data['DayOfYear_Sin'] = np.sin(2 * np.pi * data['Date'].dt.dayofyear / 365.25).round(2)
data['DayOfYear_Cos'] = np.cos(2 * np.pi * data['Date'].dt.dayofyear / 365.25).round(2)

data.drop(columns=['Year', "Month", "Day", "DayOfYear"], inplace=True)

write_log(data, "Transformacija vremenskih podataka", "weatherAusAfter9_1_1.csv")

#### 9.1.2 Podaci o vetru

U datasetu se nalazi veći broj kolona koje naznačavaju smer najjačeg vetra u određenim momentima dana, ove kolone, kao vremenski podaci, predstavljaju ordinalne kategorijske promenljive sa cikličnim redosledom. 
Ako bismo svaki smer vetar počevši od severa kodirali vrednostima počevši od jedan u smeru kazaljke na satu, dobili bismo rezultat koji će našem modelu govoriti da je severozapadni vetar najveća razlika severnome vetru, što nije tačno.
Kao i kod vremenskih podataka, da bismo rešili ovaj problem izvršićemo kružno kodiranje.

In [ ]:
wind_dir_map = {
    'N': 0, 'NNE': 22.5, 'NE': 45, 'ENE': 67.5,
    'E': 90, 'ESE': 112.5, 'SE': 135, 'SSE': 157.5,
    'S': 180, 'SSW': 202.5, 'SW': 225, 'WSW': 247.5,
    'W': 270, 'WNW': 292.5, 'NW': 315, 'NNW': 337.5
}
def wind_dir_to_sin_cos(df, col_name):
    df[col_name + '_deg'] = df[col_name].map(wind_dir_map)
    df[col_name + '_sin'] = np.sin(np.radians(df[col_name + '_deg'])).round(2)
    df[col_name + '_cos'] = np.cos(np.radians(df[col_name + '_deg'])).round(2)
    df.drop(columns=[col_name, col_name + '_deg'], inplace=True)
    return df
def convertWinds(df):
    df = wind_dir_to_sin_cos(df, 'WindDir3pm')
    df = wind_dir_to_sin_cos(df, 'WindDir9am')
    df = wind_dir_to_sin_cos(df, 'WindGustDir')
    return df
data = convertWinds(data)
write_log(data, "Transformacija podataka o vetru", "weatherAusAfter9_1_2.csv")


**Napomena o rupama u vremenskoj seriji:** u odeljku 6.4.2.1 utvrdili smo da postoje periodi kada pojedinim (ili gotovo svim) lokacijama nedostaju čitavi nizovi uzastopnih dana (najizraženije: april 2011, decembar 2012. i februar 201). Zbog toga uvodimo pomoćnu funkciju koja proverava da li razmak zaista odgovara traženom broju dana, i ako ne odgovara, vraća NaN (umesto lažne vrednosti) i takav NaN se dalje tretira identično svim ostalim nedostajućim vrednostima u odeljku 11.

### 9.2 Dodavanje novih atributa

#### 9.2.1 Diferencijalni atributi

Na osnovu domenskog znanja i logike mozemo pretpostaviti da je u meterologiji mnogo bitnija promena velicine od samih vrednosti. Na primer: ako pritisak izmedju 9 i 15 casova znatno opadne to je mnogo jaci signal da postoji verovatnoca za kisu od samih vrednost, zato uvedimo nove kolone koje ce predstavljati promenu meteroloskih velicina u intervalu od 9 do 15 casova. Takodje, mozemo dodati kolonu koja ce predstavljati dnevni raspone temperature (razliku najvise i najnize).

Imajući u vidu da i dalje imamo nedostajuce podatke i da ćemo nakon imputacije ponovo trebati da preračunamo ove kolone, kreiraćeo funkciju za računanje ovih kolona.

In [ ]:
data = calculateDiffs(data)
write_log(data, "Dodavanje diferencijalnih atributa", "weatherAusAfter9_2_1.csv")


#### 9.2.2 Pokretni proseci

Jedan od bitnih podataka kojima bismo mogli da nadomestimo veliku kolicinu nedostajucih podataka su okretni proseci, odnosno proseci određene veličine u prethodnih n dana.

In [ ]:
data = calculateRollingMeans(data)
write_log(data, "Dodavanje pokretnih proseka", "weatherAusAfter9_2_2.csv")

#### 9.2.3 Lag podaci

Jos jedan podatak koji će nam značiti u našoj budućoj analizi, pa nam je pogodno da ga imamo u svakoj vrsti, je informacija o  vrednostima odredjenih veličine pre odredjenog broja dana. Kreiraćemo funkciju za računanje ovih vrednosti.

In [ ]:
data = calculatePreviousValues(data)
write_log(data,"Dodavanje lag podataka", "weatherAusAfter9_2_3.csv")

#### 9.2.4 Funkcija za sveobuhvatno dodavanje podataka

Trenutni poziv funkcija moze kreirati vrednosti za nove atribute samo tamo gde postoje vrednosti za originalne atribute, kako cemo u nastavku vrsiti imputaciju nedostajucih podataka, javice se potreba da nakon svake faze imputacije se izvrsi ponovno kreiranje izvedenih promenjivih. 
Zbog toga cemo kreirati jednu metodu koja ce kad se pozove ponovno kreirati sve promenjljive.

#### 9.2.5 Ucitavanje podataka o stanicama

Svaka merna stanica ima različite osobine koji proazilaze iz njenih geografskih (nadmorska visina, pozicija...), ove osobine će dratično uticati na predikciju za svaku lokaciju, takođe pomoću podataka o lokaciji i nadmorskoj visini, naš model će moći zaključiti koje lokacije imaju slične osobine.
Prvo ćemo ekstrahovati sve jedinstvene nazive stanica u našem dataset-u, novodobijeni dataframe predstavlja će osnovu dalje analize podataka o stanicama.